# Plant Pathology 2021 Offline Inference Submission

This notebook is a Kaggle submission-only version of the training notebook.

- `Internet` must be set to `Off` in the Kaggle notebook editor before saving a version
- attach the competition dataset
- attach the dataset that contains `best_dinov2_small_plant_pathology_fold0.pth`
- run this notebook to generate `submission.csv`

This notebook does not train, validate, or tune thresholds. It only loads a saved checkpoint and runs test-set inference.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "This offline inference notebook requires `timm` to already exist in the Kaggle runtime. "
        "Internet installation is disabled here. If Kaggle cannot import `timm`, attach an offline dependency dataset instead."
    ) from exc

BASE_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8"
CHECKPOINT_PATH = "/kaggle/input/<your-model-dataset>/best_dinov2_small_plant_pathology_fold0.pth"
MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
IMAGE_SIZE = 518
GRID_SIZE = 2
USE_TILING_FOR_INFERENCE = True
TILE_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 2
ALL_LABELS = [
    "complex",
    "frog_eye_leaf_spot",
    "healthy",
    "powdery_mildew",
    "rust",
    "scab",
]
NUM_CLASSES = len(ALL_LABELS)

TRAIN_DIR = os.path.join(BASE_DIR, "train_images")
TEST_DIR = os.path.join(BASE_DIR, "test_images")
SAMPLE_SUBMISSION_PATH = os.path.join(BASE_DIR, "sample_submission.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = DEFAULT_NUM_WORKERS if DEVICE.type == "cuda" else 0
PIN_MEMORY = DEVICE.type == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", DEVICE)
print("Internet mode: expected OFF in Kaggle notebook settings")
print("Checkpoint path:", CHECKPOINT_PATH)
print("Tiled inference:", USE_TILING_FOR_INFERENCE)


In [ ]:
if "<your-model-dataset>" in CHECKPOINT_PATH:
    raise ValueError(
        "Update CHECKPOINT_PATH to the real Kaggle input path for your checkpoint dataset before running inference."
    )

if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Competition dataset was not found at {BASE_DIR}. Attach the Plant Pathology 2021 competition dataset first."
    )

if not Path(CHECKPOINT_PATH).exists():
    raise FileNotFoundError(
        f"Checkpoint file was not found at {CHECKPOINT_PATH}. Attach the dataset that contains best_dinov2_small_plant_pathology_fold0.pth."
    )

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_image_names = sample_submission["image"].tolist()

print("Sample submission shape:", sample_submission.shape)
print("Test image count:", len(test_image_names))
print(sample_submission.head())


In [ ]:
class PlantPathologyTestDataset(Dataset):
    def __init__(self, image_names, image_dir, transform=None, mode="tile_inference", grid_size=2):
        self.image_names = list(image_names)
        self.image_dir = image_dir
        self.transform = transform
        self.mode = mode
        self.grid_size = grid_size

        if self.mode not in {"image", "tile_inference"}:
            raise ValueError(f"Unsupported mode: {self.mode}")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = os.path.join(self.image_dir, image_name)
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_tensor = self.transform(image) if self.transform else image
        return image_tensor, image_name


def collate_full_images(batch):
    images = [item[0] for item in batch]
    image_names = [item[1] for item in batch]
    return images, image_names


def build_dataloader_kwargs(batch_size, collate_fn=None):
    kwargs = {
        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return kwargs


probe_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
DATA_CONFIG = timm.data.resolve_model_data_config(probe_model)
del probe_model

NORMALIZE_MEAN = DATA_CONFIG.get("mean", (0.485, 0.456, 0.406))
NORMALIZE_STD = DATA_CONFIG.get("std", (0.229, 0.224, 0.225))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

eval_full_image_transform = transforms.Compose([
    transforms.PILToTensor(),
])

print("Normalization mean:", NORMALIZE_MEAN)
print("Normalization std:", NORMALIZE_STD)


In [ ]:
class PlantDiseaseDINOv2Model(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None)
        if self.feature_dim is None:
            raise ValueError("Could not infer backbone feature dimension.")

        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)

        if isinstance(features, dict):
            features = next(iter(features.values()))
        elif isinstance(features, (list, tuple)):
            features = features[0]

        if features.ndim == 3:
            features = features[:, 0]

        return self.head(features)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def gpu_tile_and_resize(image, grid_size, image_size, mean, std):
    _, height, width = image.shape
    cropped_height = height - (height % grid_size)
    cropped_width = width - (width % grid_size)
    if cropped_height == 0 or cropped_width == 0:
        raise ValueError(f"Image size {(height, width)} is too small for grid_size={grid_size}.")

    tile_height = cropped_height // grid_size
    tile_width = cropped_width // grid_size
    cropped_image = image[:, :cropped_height, :cropped_width]
    tile_batch = cropped_image.unfold(1, tile_height, tile_height).unfold(2, tile_width, tile_width)
    tile_batch = tile_batch.permute(1, 2, 0, 3, 4).reshape(-1, image.size(0), tile_height, tile_width)
    tile_batch = F.interpolate(
        tile_batch.float().div(255.0),
        size=(image_size, image_size),
        mode="bicubic",
        align_corners=False,
    )
    mean_tensor = tile_batch.new_tensor(mean).view(1, -1, 1, 1)
    std_tensor = tile_batch.new_tensor(std).view(1, -1, 1, 1)
    tile_batch = tile_batch.sub(mean_tensor).div(std_tensor)
    return tile_batch


def evaluate_image_level(model, loader, device, aggregation="max", tiled=True):
    model.eval()
    all_probabilities = []
    all_names = []

    with torch.no_grad():
        progress = tqdm(loader, desc="Test inference", leave=False)
        for batch in progress:
            if tiled:
                images, image_names = batch
                batch_size = len(images)
                tile_batches = [
                    gpu_tile_and_resize(
                        image=image.to(device, non_blocking=True),
                        grid_size=GRID_SIZE,
                        image_size=IMAGE_SIZE,
                        mean=NORMALIZE_MEAN,
                        std=NORMALIZE_STD,
                    )
                    for image in images
                ]
                flat_tiles = torch.cat(tile_batches, dim=0)
                num_tiles = GRID_SIZE ** 2

                with amp.autocast(device_type=device.type, enabled=USE_AMP):
                    logits = model(flat_tiles)

                probs = torch.sigmoid(logits).float().view(batch_size, num_tiles, NUM_CLASSES)
                if aggregation == "max":
                    image_probs = probs.max(dim=1).values
                elif aggregation == "mean":
                    image_probs = probs.mean(dim=1)
                else:
                    raise ValueError(f"Unsupported aggregation method: {aggregation}")

                all_probabilities.append(image_probs.cpu().numpy())
                all_names.extend(list(image_names))
            else:
                images, image_names = batch
                images = images.to(device, non_blocking=True)
                with amp.autocast(device_type=device.type, enabled=USE_AMP):
                    logits = model(images)
                probs = torch.sigmoid(logits).float().cpu().numpy()
                all_probabilities.append(probs)
                all_names.extend(list(image_names))

    return {
        "probs": np.concatenate(all_probabilities, axis=0),
        "image_names": all_names,
    }


def probs_to_label(prob_row, thresholds):
    selected = [ALL_LABELS[i] for i, prob in enumerate(prob_row) if prob >= thresholds[i]]
    if not selected:
        selected = [ALL_LABELS[int(np.argmax(prob_row))]]
    return " ".join(selected)


In [ ]:
test_dataset = PlantPathologyTestDataset(
    image_names=test_image_names,
    image_dir=TEST_DIR,
    transform=eval_full_image_transform if USE_TILING_FOR_INFERENCE else eval_transform,
    mode="tile_inference" if USE_TILING_FOR_INFERENCE else "image",
    grid_size=GRID_SIZE,
)

test_loader = DataLoader(
    test_dataset,
    **build_dataloader_kwargs(
        batch_size=TILE_BATCH_SIZE if USE_TILING_FOR_INFERENCE else EVAL_BATCH_SIZE,
        collate_fn=collate_full_images if USE_TILING_FOR_INFERENCE else None,
    ),
)

submission_model = PlantDiseaseDINOv2Model(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
except Exception as exc:
    raise RuntimeError(
        f"Failed to load checkpoint from {CHECKPOINT_PATH} with weights_only=True."
    ) from exc

if "model_state" not in checkpoint:
    raise KeyError(
        "Checkpoint is missing `model_state`. This notebook expects the saved best-fold checkpoint produced by the training notebook."
    )

thresholds = np.array(checkpoint.get("thresholds", [0.5] * NUM_CLASSES), dtype=np.float32)
aggregation_method = checkpoint.get("aggregation_method", "max")
checkpoint_stage = checkpoint.get("stage", "unknown")

if thresholds.shape != (NUM_CLASSES,):
    raise ValueError(
        f"Checkpoint thresholds shape {thresholds.shape} does not match NUM_CLASSES={NUM_CLASSES}."
    )

try:
    unwrap_model(submission_model).load_state_dict(checkpoint["model_state"])
except RuntimeError as exc:
    raise RuntimeError(
        "Checkpoint weights do not match PlantDiseaseDINOv2Model. Confirm MODEL_NAME and classifier head match the training notebook."
    ) from exc

submission_model.eval()

print("Checkpoint stage:", checkpoint_stage)
print("Aggregation method:", aggregation_method)
print("Thresholds:")
for label, threshold in zip(ALL_LABELS, thresholds):
    print(f"  {label:>20s}: {float(threshold):.2f}")


In [ ]:
test_outputs = evaluate_image_level(
    model=submission_model,
    loader=test_loader,
    device=DEVICE,
    aggregation=aggregation_method,
    tiled=USE_TILING_FOR_INFERENCE,
)

test_probabilities = test_outputs["probs"]
predicted_image_names = test_outputs["image_names"]
predicted_labels = [probs_to_label(prob_row, thresholds) for prob_row in test_probabilities]

submission = pd.DataFrame({
    "image": predicted_image_names,
    "labels": predicted_labels,
})

submission = submission.sort_values("image").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

if len(submission) != len(sample_submission):
    raise ValueError(
        f"Submission row count {len(submission)} does not match sample submission row count {len(sample_submission)}."
    )

missing_columns = set(sample_submission.columns) - set(submission.columns)
if missing_columns:
    raise ValueError(f"Submission is missing expected columns: {sorted(missing_columns)}")

empty_rows = int((submission["labels"].str.len() == 0).sum())

print("submission.csv saved!")
print("\nFirst 5 rows:")
print(submission.head())
print("\nShape:", submission.shape)
print("Rows with empty labels:", empty_rows)
